In [ ]:
"""
kaggle_train.py
Self-contained Kaggle-ready training script & notebook for Adaptive Business Entity Resolution.

Designed for execution on Kaggle GPU/CPU instances:
- Automatically detects dataset in /kaggle/input/** or local paths.
- Self-contained: embeds all adaptive parsing, dynamic IDF, blocking, and feature extraction.
- Fully vectorized and multi-threaded (LightGBM on multi-core CPU).
- Strictly preserves physical similarity properties via monotonic constraints.
- Macro F0.5 threshold optimization directly calibrated for the competition metric.
- Saves `adaptive_matcher.pkl` and `adaptive_matcher_meta.json` directly to /kaggle/working/.
"""

In [ ]:
import os
import sys
import time
import math
import json
import re
import pickle
import unicodedata
from collections import Counter, defaultdict, namedtuple
from typing import Dict, List, Optional, Set, Tuple, Union
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

---------------------------------------------------------------------------
1. ADAPTIVE PARSING, REPRESENTATIONS & FEATURES (SELF-CONTAINED)
---------------------------------------------------------------------------

In [ ]:
RE_NON_ALPHANUM = re.compile(r'[^a-z0-9\s]+')
RE_DIGITS = re.compile(r'\b\d+\b')

In [ ]:
FEATURE_NAMES = [
    'name_jw',
    'name_jw_compact',
    'name_token_sort',
    'name_token_set',
    'name_char_3gram_jaccard',
    'name_char_4gram_jaccard',
    'name_idf_weighted_jaccard',
    'name_exact_compact',
    'addr_token_sort',
    'addr_token_set',
    'addr_char_3gram_jaccard',
    'addr_number_jaccard',
    'addr_number_conflict',
    'addr_number_match_count',
    'addr_is_null',
    'addr_both_null',
    'is_source2'
]

In [ ]:
MONOTONE_CONSTRAINTS = [
    1,   # name_jw
    1,   # name_jw_compact
    1,   # name_token_sort
    1,   # name_token_set
    1,   # name_char_3gram_jaccard
    1,   # name_char_4gram_jaccard
    1,   # name_idf_weighted_jaccard
    1,   # name_exact_compact
    1,   # addr_token_sort
    1,   # addr_token_set
    1,   # addr_char_3gram_jaccard
    1,   # addr_number_jaccard
    -1,  # addr_number_conflict
    1,   # addr_number_match_count
    0,   # addr_is_null
    0,   # addr_both_null
    0    # is_source2
]

In [ ]:
ParsedRecord = namedtuple(
    'ParsedRecord',
    [
        'name_norm',
        'name_sig',
        'name_words',
        'name_words_set',
        'name_3g',
        'name_4g',
        'addr_norm',
        'addr_words',
        'addr_nums',
        'addr_3g',
        'is_null',
        'first_street_word'
    ]
)

In [ ]:
def normalize_text(text: Optional[str]) -> str:
    """Universal NFKD Unicode normalization."""
    if text is None:
        return ''
    s = str(text)
    if not s or s == 'None' or s == 'nan':
        return ''
    s = s.replace('&', ' and ').replace('+', ' and ').replace('@', ' at ')
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c)).lower()
    return ' '.join(RE_NON_ALPHANUM.sub(' ', s).split())

In [ ]:
def _extract_char_ngrams(s: str, n: int) -> Set[str]:
    """Extract character n-grams from string."""
    if not s:
        return set()
    if len(s) < n:
        return {s}
    return {s[i:i + n] for i in range(len(s) - n + 1)}

In [ ]:
def _extract_first_street_word(addr_words: List[str]) -> str:
    """Extract first distinctive non-numeric street token adjacent to house number."""
    digits = [w for w in addr_words if w.isdigit()]
    if not digits:
        return ''
    first_digit = digits[0]
    try:
        idx = addr_words.index(first_digit)
        after = [w for w in addr_words[idx + 1:] if not w.isdigit() and len(w) >= 3]
        if after:
            return after[0]
        before = [w for w in addr_words[:idx] if not w.isdigit() and len(w) >= 3]
        if before:
            return before[-1]
    except ValueError:
        pass
    return ''

In [ ]:
def parse_record(name: Optional[str], addr: Optional[str]) -> ParsedRecord:
    """High-speed pre-parsing function for business records."""
    name_norm = normalize_text(name)
    if name_norm:
        name_words = name_norm.split()
        name_words_set = set(name_words)
        name_sig = ''.join(name_words)
        name_3g = _extract_char_ngrams(name_norm, 3)
        name_4g = _extract_char_ngrams(name_norm, 4)
    else:
        name_words = []
        name_words_set = set()
        name_sig = ''
        name_3g = set()
        name_4g = set()

    addr_norm = normalize_text(addr)
    if addr_norm:
        addr_words = addr_norm.split()
        addr_nums = {w for w in addr_words if w.isdigit()}
        addr_3g = _extract_char_ngrams(addr_norm, 3)
        is_null = 0.0
        street = _extract_first_street_word(addr_words)
    else:
        addr_words = []
        addr_nums = set()
        addr_3g = set()
        is_null = 1.0
        street = ''

    return ParsedRecord(
        name_norm=name_norm,
        name_sig=name_sig,
        name_words=name_words,
        name_words_set=name_words_set,
        name_3g=name_3g,
        name_4g=name_4g,
        addr_norm=addr_norm,
        addr_words=addr_words,
        addr_nums=addr_nums,
        addr_3g=addr_3g,
        is_null=is_null,
        first_street_word=street
    )

In [ ]:
class CorpusIDF:
    def __init__(self, idf: Dict[str, float], stopwords: Set[str], df: Dict[str, int], default_idf: float, n_records: int):
        self.idf = idf
        self.stopwords = stopwords
        self.df = df
        self.default_idf = default_idf
        self.n_records = n_records

In [ ]:
def compute_corpus_idf(records_parsed: List[ParsedRecord]) -> CorpusIDF:
    """Computes document frequency DF and dynamic stopwords for the target pool."""
    n_records = len(records_parsed)
    if n_records == 0:
        return CorpusIDF({}, set(), {}, 0.0, 0)

    df_counter = Counter()
    for r in records_parsed:
        df_counter.update(r.name_words_set)

    df_threshold = 0.005 * n_records
    stopwords = {w for w, count in df_counter.items() if count > df_threshold}
    top_100 = {w for w, count in df_counter.most_common(100) if count >= 2}
    stopwords.update(top_100)

    default_idf = math.log(1.0 + float(n_records))
    idf_dict = {
        w: math.log(1.0 + float(n_records) / (1.0 + float(c)))
        for w, c in df_counter.items()
    }

    return CorpusIDF(
        idf=idf_dict,
        stopwords=stopwords,
        df=dict(df_counter),
        default_idf=default_idf,
        n_records=n_records
    )

In [ ]:
def compute_parsed_features(
    p1: ParsedRecord,
    p2: ParsedRecord,
    is_s2: Union[bool, int, float],
    idf_dict: Optional[Dict[str, float]] = None,
    default_idf: Optional[float] = None
) -> List[float]:
    """Extract relative similarity features strictly bounded in [0, 1]."""
    # Name features
    f_name_jw = JaroWinkler.similarity(p1.name_norm, p2.name_norm)
    f_name_jw_compact = JaroWinkler.similarity(p1.name_sig, p2.name_sig)
    f_name_token_sort = fuzz.token_sort_ratio(p1.name_norm, p2.name_norm) / 100.0
    f_name_token_set = fuzz.token_set_ratio(p1.name_norm, p2.name_norm) / 100.0

    u3 = len(p1.name_3g | p2.name_3g)
    f_name_char_3gram = len(p1.name_3g & p2.name_3g) / u3 if u3 > 0 else 0.0

    u4 = len(p1.name_4g | p2.name_4g)
    f_name_char_4gram = len(p1.name_4g & p2.name_4g) / u4 if u4 > 0 else 0.0

    union_words = p1.name_words_set | p2.name_words_set
    if not union_words:
        f_name_idf_jaccard = 0.0
    else:
        inter_words = p1.name_words_set & p2.name_words_set
        if not inter_words:
            f_name_idf_jaccard = 0.0
        elif inter_words == union_words:
            f_name_idf_jaccard = 1.0
        else:
            def_idf = default_idf if default_idf is not None else 1.0
            if idf_dict:
                sum_inter = sum(idf_dict.get(w, def_idf) for w in inter_words)
                sum_union = sum(idf_dict.get(w, def_idf) for w in union_words)
            else:
                sum_inter = float(len(inter_words))
                sum_union = float(len(union_words))
            f_name_idf_jaccard = min(max(sum_inter / sum_union, 0.0), 1.0) if sum_union > 0 else 0.0

    f_name_exact_compact = 1.0 if (p1.name_sig and p1.name_sig == p2.name_sig) else 0.0

    # Address features
    null1 = p1.is_null >= 0.5
    null2 = p2.is_null >= 0.5
    f_addr_is_null = 1.0 if null2 else 0.0
    f_addr_both_null = 1.0 if (null1 and null2) else 0.0

    if null1 or null2:
        f_addr_token_sort = 0.0
        f_addr_token_set = 0.0
        f_addr_char_3gram = 0.0
        f_addr_number_jaccard = 0.0
        f_addr_number_conflict = 0.0
        f_addr_number_match_count = 0.0
    else:
        f_addr_token_sort = fuzz.token_sort_ratio(p1.addr_norm, p2.addr_norm) / 100.0
        f_addr_token_set = fuzz.token_set_ratio(p1.addr_norm, p2.addr_norm) / 100.0

        u_addr3 = len(p1.addr_3g | p2.addr_3g)
        f_addr_char_3gram = len(p1.addr_3g & p2.addr_3g) / u_addr3 if u_addr3 > 0 else 0.0

        num_inter = len(p1.addr_nums & p2.addr_nums)
        num_union = len(p1.addr_nums | p2.addr_nums)

        f_addr_number_jaccard = num_inter / num_union if num_union > 0 else 0.0
        f_addr_number_conflict = 1.0 if (len(p1.addr_nums) > 0 and len(p2.addr_nums) > 0 and num_inter == 0) else 0.0
        f_addr_number_match_count = min(num_inter / 3.0, 1.0)

    f_is_source2 = 1.0 if bool(is_s2) else 0.0

    return [
        f_name_jw,
        f_name_jw_compact,
        f_name_token_sort,
        f_name_token_set,
        f_name_char_3gram,
        f_name_char_4gram,
        f_name_idf_jaccard,
        f_name_exact_compact,
        f_addr_token_sort,
        f_addr_token_set,
        f_addr_char_3gram,
        f_addr_number_jaccard,
        f_addr_number_conflict,
        f_addr_number_match_count,
        f_addr_is_null,
        f_addr_both_null,
        f_is_source2
    ]

In [ ]:
class AdaptiveInvertedIndex:
    def __init__(self, max_posting: int = 30, max_candidates: int = 15, stopwords: Optional[Set[str]] = None, df_dict: Optional[Dict[str, int]] = None):
        self.index = defaultdict(list)
        self.max_posting = max_posting
        self.max_candidates = max_candidates
        self.stopwords = stopwords if stopwords is not None else set()
        self.df = df_dict if df_dict is not None else {}

    def add_records(self, parsed_records: List[ParsedRecord], is_s2_flag: int):
        sw = self.stopwords
        df = self.df
        flag = 1 if is_s2_flag == 1 else 0

        for i, r in enumerate(parsed_records):
            sig = r.name_sig
            if len(sig) >= 5:
                k6 = f"p6_{sig[:6]}"
                lst6 = self.index[k6]
                if len(lst6) < self.max_posting:
                    lst6.append((flag, i))
                if len(sig) >= 8:
                    k8 = f"p8_{sig[:8]}"
                    lst8 = self.index[k8]
                    if len(lst8) < self.max_posting:
                        lst8.append((flag, i))

            for w in r.name_words_set:
                if len(w) >= 3 and (w not in sw):
                    token_df = df.get(w, 0)
                    if 2 <= token_df <= 300:
                        k_tok = f"t_{w}"
                        lst_tok = self.index[k_tok]
                        if len(lst_tok) < self.max_posting:
                            lst_tok.append((flag, i))

            if (r.is_null < 0.5) and r.addr_nums and r.first_street_word:
                digit = sorted(list(r.addr_nums))[0]
                k_addr = f"num_{digit}_{r.first_street_word}"
                lst_addr = self.index[k_addr]
                if len(lst_addr) < self.max_posting:
                    lst_addr.append((flag, i))

    def query(self, parsed_s1: ParsedRecord) -> List[Tuple[int, int]]:
        sw = self.stopwords
        df = self.df
        cand_scores = Counter()

        sig = parsed_s1.name_sig
        if len(sig) >= 8 and f"p8_{sig[:8]}" in self.index:
            for item in self.index[f"p8_{sig[:8]}"]:
                cand_scores[item] += 3
        elif len(sig) >= 5 and f"p6_{sig[:6]}" in self.index:
            for item in self.index[f"p6_{sig[:6]}"]:
                cand_scores[item] += 2

        toks = [w for w in parsed_s1.name_words_set if len(w) >= 3 and (w not in sw) and (2 <= df.get(w, 0) <= 300)]
        toks.sort(key=lambda w: df.get(w, 0))
        for w in toks[:3]:
            for item in self.index.get(f"t_{w}", []):
                cand_scores[item] += 1

        if (parsed_s1.is_null < 0.5) and parsed_s1.addr_nums and parsed_s1.first_street_word:
            digit = sorted(list(parsed_s1.addr_nums))[0]
            k_addr = f"num_{digit}_{parsed_s1.first_street_word}"
            for item in self.index.get(k_addr, []):
                cand_scores[item] += 2

        return [c for c, _ in cand_scores.most_common(self.max_candidates)]

---------------------------------------------------------------------------
2. DATA DISCOVERY & TRAINING PIPELINE
---------------------------------------------------------------------------

In [ ]:
def detect_kaggle_or_local_paths():
    """Auto-detect data directory on Kaggle or local environments."""
    candidates = [
        "/kaggle/input",
        "student_resource/dataset/train",
        "dataset/train",
        r"C:\Users\anmol\OneDrive\Desktop\Work\ML_challenge\student_resource\dataset\train",
        "../input"
    ]

    # Search for train_ground_truth.tsv
    for root_cand in candidates:
        if os.path.exists(root_cand):
            for root, _, files in os.walk(root_cand):
                if "train_ground_truth.tsv" in files and "train_source1.tsv" in files:
                    print(f"[Path Detector] Found training files in: {root}")
                    return root

    raise FileNotFoundError("Could not auto-detect training dataset directory.")

In [ ]:
def compute_macro_f05(ground_truth: Dict[str, Set[str]], predictions: Dict[str, Set[str]]) -> float:
    total_score = 0.0
    n = len(ground_truth)
    if n == 0:
        return 0.0

    for sid, y_true in ground_truth.items():
        y_pred = predictions.get(sid, set())
        len_true = len(y_true)
        len_pred = len(y_pred)

        if len_true == 0:
            if len_pred == 0:
                total_score += 1.0
            continue

        if len_pred == 0:
            continue

        tp = len(y_true & y_pred)
        if tp == 0:
            continue

        score = (5.0 * tp) / (float(len_pred) + 4.0 * float(len_true))
        total_score += score

    return total_score / float(n)

In [ ]:
def run_training_pipeline(
    data_dir: str,
    output_dir: str,
    sample_size: int = 50000,
    random_state: int = 42
):
    print("=" * 70)
    print("KAGGLE ADAPTIVE BUSINESS ENTITY RESOLUTION PIPELINE")
    print("=" * 70)
    print(f"Data Directory : {data_dir}")
    print(f"Output Directory: {output_dir}")
    print(f"Sample Size    : {sample_size if sample_size > 0 else 'Full'}")
    print("=" * 70)

    os.makedirs(output_dir, exist_ok=True)
    t0 = time.time()

    # Load S1 and Ground Truth
    s1_path = os.path.join(data_dir, "train_source1.tsv")
    gt_path = os.path.join(data_dir, "train_ground_truth.tsv")
    s2_path = os.path.join(data_dir, "train_source2.tsv")
    s3_path = os.path.join(data_dir, "train_source3.tsv")

    print(f"Loading Source 1 records...")
    if sample_size > 0:
        s1 = pl.read_csv(s1_path, separator="\t", n_rows=sample_size)
    else:
        s1 = pl.read_csv(s1_path, separator="\t")
    print(f"Loaded {len(s1):,} S1 entities.")

    s1_set = set(s1["entity_id"])
    gt = pl.read_csv(gt_path, separator="\t")
    gt_sample = gt.filter(pl.col("source1_entity_id").is_in(s1_set))

    gt_map: Dict[str, Set[str]] = {}
    needed_s2: Set[str] = set()
    needed_s3: Set[str] = set()
    singletons = 0

    for r in gt_sample.iter_rows(named=True):
        sid = r["source1_entity_id"]
        m = r["matched_entity_ids"]
        if m and str(m).strip() != "" and str(m) != "None" and str(m) != "nan":
            targets = set(x.strip() for x in str(m).split(",") if x.strip())
            gt_map[sid] = targets
            for tid in targets:
                if tid.startswith("S2-"):
                    needed_s2.add(tid)
                elif tid.startswith("S3-"):
                    needed_s3.add(tid)
        else:
            gt_map[sid] = set()
            singletons += 1

    for sid in s1["entity_id"]:
        if sid not in gt_map:
            gt_map[sid] = set()
            singletons += 1

    print(f"GT: Matched={len(s1)-singletons:,}, Singletons={singletons:,} ({singletons/len(s1)*100:.2f}%)")
    print(f"Scanning target pools for {len(needed_s2):,} S2 and {len(needed_s3):,} S3...")

    s2_df = pl.scan_csv(s2_path, separator="\t").filter(pl.col("entity_id").is_in(needed_s2)).collect()
    s3_df = pl.scan_csv(s3_path, separator="\t").filter(pl.col("entity_id").is_in(needed_s3)).collect()

    # Pre-parse records
    print(f"Pre-parsing records...")
    s1_ids = s1["entity_id"].to_list()
    s1_parsed = [parse_record(n, a) for n, a in zip(s1["business_name"], s1["business_address"])]
    s2_ids = s2_df["entity_id"].to_list()
    s2_parsed = [parse_record(n, a) for n, a in zip(s2_df["business_name"], s2_df["business_address"])]
    s3_ids = s3_df["entity_id"].to_list()
    s3_parsed = [parse_record(n, a) for n, a in zip(s3_df["business_name"], s3_df["business_address"])]

    # Corpus IDF & Inverted Index
    target_parsed = s2_parsed + s3_parsed
    corpus = compute_corpus_idf(target_parsed)
    index = AdaptiveInvertedIndex(max_posting=30, stopwords=corpus.stopwords, df_dict=corpus.df)
    index.add_records(s2_parsed, 1)
    index.add_records(s3_parsed, 0)

    s2_map = {eid: p for eid, p in zip(s2_ids, s2_parsed)}
    s3_map = {eid: p for eid, p in zip(s3_ids, s3_parsed)}

    # S1 Stratified Split (No entity leakage)
    s1_indices = np.arange(len(s1_ids))
    is_singleton = np.array([1 if len(gt_map[sid]) == 0 else 0 for sid in s1_ids])
    train_idx, val_idx = train_test_split(s1_indices, test_size=0.2, random_state=random_state, stratify=is_singleton)

    val_s1_ids = set(s1_ids[i] for i in val_idx)
    print(f"Split S1: {len(train_idx):,} Train | {len(val_idx):,} Validation")

    # Generate Train Pairs
    print("Generating train feature vectors...")
    X_train, y_train = [], []
    for i in train_idx:
        sid = s1_ids[i]
        p1 = s1_parsed[i]
        true_targets = gt_map.get(sid, set())
        for tid in true_targets:
            if tid in s2_map:
                X_train.append(compute_parsed_features(p1, s2_map[tid], True, corpus.idf, corpus.default_idf))
                y_train.append(1)
            elif tid in s3_map:
                X_train.append(compute_parsed_features(p1, s3_map[tid], False, corpus.idf, corpus.default_idf))
                y_train.append(1)

        cands = index.query(p1)
        for is_s2, row_idx in cands:
            tid = s2_ids[row_idx] if is_s2 == 1 else s3_ids[row_idx]
            if tid not in true_targets:
                p2 = s2_parsed[row_idx] if is_s2 == 1 else s3_parsed[row_idx]
                X_train.append(compute_parsed_features(p1, p2, is_s2 == 1, corpus.idf, corpus.default_idf))
                y_train.append(0)

    # Generate Val Pairs
    X_val, y_val = [], []
    val_cand_pairs = []
    for i in val_idx:
        sid = s1_ids[i]
        p1 = s1_parsed[i]
        true_targets = gt_map.get(sid, set())
        for tid in true_targets:
            if tid in s2_map:
                X_val.append(compute_parsed_features(p1, s2_map[tid], True, corpus.idf, corpus.default_idf))
                y_val.append(1)
            elif tid in s3_map:
                X_val.append(compute_parsed_features(p1, s3_map[tid], False, corpus.idf, corpus.default_idf))
                y_val.append(1)

        cands = index.query(p1)
        for is_s2, row_idx in cands:
            tid = s2_ids[row_idx] if is_s2 == 1 else s3_ids[row_idx]
            p2 = s2_parsed[row_idx] if is_s2 == 1 else s3_parsed[row_idx]
            feats = compute_parsed_features(p1, p2, is_s2 == 1, corpus.idf, corpus.default_idf)
            val_cand_pairs.append((sid, tid, feats))
            if tid not in true_targets:
                X_val.append(feats)
                y_val.append(0)

    X_train_arr = np.array(X_train, dtype=np.float32)
    y_train_arr = np.array(y_train, dtype=np.int32)
    X_val_arr = np.array(X_val, dtype=np.float32)
    y_val_arr = np.array(y_val, dtype=np.int32)

    print(f"Train Pairs: {len(X_train_arr):,} (Pos: {sum(y_train_arr):,}, Neg: {len(y_train_arr)-sum(y_train_arr):,})")
    print(f"Val Pairs  : {len(X_val_arr):,} (Pos: {sum(y_val_arr):,}, Neg: {len(y_val_arr)-sum(y_val_arr):,})")

    # Train LightGBM Classifier
    print("\nTraining LightGBM Classifier with Monotonic Constraints...")
    model = lgb.LGBMClassifier(
        objective="binary",
        metric="binary_logloss",
        n_estimators=300,
        learning_rate=0.08,
        num_leaves=31,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        monotone_constraints=MONOTONE_CONSTRAINTS,
        random_state=random_state,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(
        X_train_arr,
        y_train_arr,
        eval_set=[(X_val_arr, y_val_arr)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )

    val_probs = model.predict_proba(X_val_arr)[:, 1]
    val_auc = roc_auc_score(y_val_arr, val_probs)
    print(f"Validation Pairwise ROC-AUC: {val_auc:.5f}")

    # Feature Importances
    print("\n--- FEATURE IMPORTANCES ---")
    importances = model.feature_importances_
    for fname, imp in sorted(zip(FEATURE_NAMES, importances), key=lambda x: x[1], reverse=True):
        print(f"  {fname:28s}: {imp:6d}")

    # Threshold Optimization on Macro F0.5
    print("\nOptimizing Decision Threshold for Macro F0.5...")
    val_X_pairs = np.array([p[2] for p in val_cand_pairs], dtype=np.float32)
    booster = model.booster_
    pair_probs = booster.predict(val_X_pairs)

    val_gt_dict = {sid: gt_map[sid] for sid in val_s1_ids}
    thresholds = np.linspace(0.40, 0.95, 29)
    best_theta = 0.80
    best_f05 = -1.0
    best_prec, best_rec = 0.0, 0.0

    for theta in thresholds:
        best_assignment = {}
        for (sid, cid, _), prob in zip(val_cand_pairs, pair_probs):
            if prob >= theta:
                if cid not in best_assignment or prob > best_assignment[cid][0]:
                    best_assignment[cid] = (prob, sid)

        s1_preds = {sid: set() for sid in val_s1_ids}
        for cid, (prob, sid) in best_assignment.items():
            s1_preds[sid].add(cid)

        f05 = compute_macro_f05(val_gt_dict, s1_preds)

        total_tp, total_pred, total_true = 0, 0, 0
        for sid, y_true in val_gt_dict.items():
            y_pred = s1_preds[sid]
            total_tp += len(y_true & y_pred)
            total_pred += len(y_pred)
            total_true += len(y_true)

        prec = total_tp / total_pred if total_pred > 0 else 1.0
        rec = total_tp / total_true if total_true > 0 else 0.0

        if f05 > best_f05:
            best_f05 = f05
            best_theta = float(theta)
            best_prec = prec
            best_rec = rec

    print(f"\n[OPTIMIZATION RESULT] Optimal Threshold: theta* = {best_theta:.2f}")
    print(f"  Validation Macro F0.5 : {best_f05:.4f}")
    print(f"  Validation Precision  : {best_prec:.4f}")
    print(f"  Validation Recall     : {best_rec:.4f}")

    # Benchmark Speed
    X_bench = np.random.uniform(0.0, 1.0, size=(200000, len(FEATURE_NAMES))).astype(np.float32)
    _ = booster.predict(X_bench[:1000])
    t_bench = time.time()
    _ = booster.predict(X_bench)
    speed = 200000 / (time.time() - t_bench)
    print(f"CPU Prediction Throughput: {speed:,.0f} predictions/sec")

    # Save artifacts
    model_path = os.path.join(output_dir, "adaptive_matcher.pkl")
    meta_path = os.path.join(output_dir, "adaptive_matcher_meta.json")

    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"Saved trained model: {model_path}")

    meta = {
        "model_type": "LightGBM Classifier",
        "optimal_threshold": round(best_theta, 4),
        "validation_macro_f05": round(best_f05, 4),
        "validation_roc_auc": round(val_auc, 5),
        "validation_precision": round(best_prec, 4),
        "validation_recall": round(best_rec, 4),
        "features": FEATURE_NAMES,
        "monotone_constraints": MONOTONE_CONSTRAINTS,
        "prediction_speed_rows_per_sec": int(speed),
        "sample_size": sample_size
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved metadata: {meta_path}")

    print("\n" + "=" * 70)
    print(f"KAGGLE TRAINING PIPELINE COMPLETE IN {time.time()-t0:.2f}s!")
    print("=" * 70)

In [ ]:
def main():
    try:
        data_dir = detect_kaggle_or_local_paths()
    except FileNotFoundError:
        data_dir = "student_resource/dataset/train"

    # Default to /kaggle/working if on kaggle
    if os.path.exists("/kaggle/working"):
        output_dir = "/kaggle/working"
    else:
        output_dir = os.path.join(os.path.dirname(__file__), "models")

    run_training_pipeline(
        data_dir=data_dir,
        output_dir=output_dir,
        sample_size=50000,
        random_state=42
    )

In [ ]:
if __name__ == "__main__":
    main()